In [1]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import functional as F
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML
from recommenders.matrix_dataset import MatrixDataset
from recommenders.lmf import LogisticMatrixFactorization

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    device = mps_device
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print ("MPS device not found.")

if device == torch.device("cuda"):
    dtype = torch.float32
    print("Using CUDA.")
elif device == torch.device("cpu"):
    dtype = torch.float64
    print("Using CPU.")
elif device == torch.device("mps"):
    dtype = torch.float32
    print("Using MPS.")

# device = "cpu"
# dtype = torch.float32

tensor([1.], device='mps:0')
Using MPS.


# Import data

In [3]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 1, "long_term": 1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,1.00,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.98,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.96,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.94,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.92,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [4]:
df_matrix_mf = df.copy()
df_matrix_mf.loc[df_matrix_mf["type"] == "liked_track", "affinity"] = 0.5
df_matrix_mf.loc[df_matrix_mf["type"] == "playlist", "affinity"] = 0.3

used_types = ["top_track", "liked_track", "playlist"]
used_types = ["top_track"]
df_matrix_mf = df_matrix_mf[df["type"].isin(used_types)]
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]
df_matrix_mf["affinity"] *= 100

In [5]:
matrix_mf = MatrixDataset(df_matrix_mf, "username", "id", "affinity", device=device, dtype=dtype)
user_interacted_with = matrix_mf.R
user_interacted_with

tensor([[ 0.,  0., 62.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0., 84.],
        ...,
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [44., 60.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ..., 92.,  0.,  0.]], device='mps:0')

In [6]:
user_interacted_with.shape

torch.Size([8, 923])

In [7]:
alpha = matrix_mf.compute_alpha().item()
user_interacted_with *= alpha
alpha

0.10527777671813965

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import List, Optional


class MultiVAE(nn.Module):
    """
    Variational Autoencoder with Multinomial Likelihood (Multi-VAE).

    This VAE is designed for collaborative filtering tasks. The architecture
    comprises of encoder (q-network) and decoder (p-network) layers. The encoder
    produces a latent representation, and the decoder reconstructs the input data.

    Attributes:
        encoder_dims (List[int]): Dimensions for the encoder layers.
        decoder_dims (List[int]): Dimensions for the decoder layers.
        dropout_rate (float): Dropout rate for regularization.

    References:
        [1] Liang, Dawen, et al. "Variational autoencoders for collaborative filtering."
            Proceedings of the 2018 World Wide Web Conference. 2018.
        [2] Variational autoencoders for collaborative filtering
            by @dawenl.
            https://github.com/dawenl/vae_cf
        [3] Variational Autoencoders for Collaborative Filtering - Implementation in PyTorch
            by @younggyoseo.
            https://github.com/younggyoseo/vae-cf-pytorch
    """

    def __init__(
        self,
        encoder_dims: List[int],
        decoder_dims: Optional[List[int]] = None,
        dropout_rate: float = 0.5,
        device: torch.device = torch.device("cpu"),
        dtype: torch.dtype = torch.float32,
    ) -> None:
        super(MultiVAE, self).__init__()
        self.device = device
        self.dtype = dtype

        self.encoder_dims = encoder_dims
        self.decoder_dims = decoder_dims if decoder_dims else encoder_dims[::-1]

        assert (
            self.decoder_dims[0] == encoder_dims[-1]
        ), "Output dimension of encoder must match input dimension of decoder."
        assert (
            self.decoder_dims[-1] == encoder_dims[0]
        ), "Latent dimension mismatch between encoder and decoder."

        # Modify the last dimension of encoder for mean and variance
        modified_encoder_dims = self.encoder_dims[:-1] + [self.encoder_dims[-1] * 2]
        self.encoder_layers = nn.ModuleList(
            [
                nn.Linear(in_features, out_features)
                for in_features, out_features in zip(
                    modified_encoder_dims[:-1], modified_encoder_dims[1:]
                )
            ]
        )
        self.decoder_layers = nn.ModuleList(
            [
                nn.Linear(in_features, out_features)
                for in_features, out_features in zip(
                    self.decoder_dims[:-1], self.decoder_dims[1:]
                )
            ]
        )

        self.dropout = nn.Dropout(dropout_rate)
        self.initialize_weights()
        self.to(device=device, dtype=dtype)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        mu, logvar = self.encode(input)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def encode(self, input: torch.Tensor) -> torch.Tensor:
        h = F.normalize(input)
        h = self.dropout(h)

        for i, layer in enumerate(self.encoder_layers):
            h = layer(h)
            if i != len(self.encoder_layers) - 1:
                h = torch.tanh(h)
            else:
                mu = h[:, : self.encoder_dims[-1]]
                logvar = h[:, self.encoder_dims[-1] :]
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return eps.mul(std).add_(mu)
        else:
            return mu

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        h = z
        for i, layer in enumerate(self.decoder_layers):
            h = layer(h)
            if i != len(self.decoder_layers) - 1:
                h = torch.tanh(h)
        return h

    def initialize_weights(self) -> None:
        for layer in self.encoder_layers + self.decoder_layers:
            size = layer.weight.size()
            fan_out, fan_in = size[0], size[1]
            std = np.sqrt(2.0 / (fan_in + fan_out))
            layer.weight.data.normal_(0.0, std)
            layer.bias.data.normal_(0.0, 0.001)

    def loss(
        self,
        recon_x: torch.Tensor,
        x: torch.Tensor,
        mu: torch.Tensor,
        logvar: torch.Tensor,
        anneal: float = 1.0,
    ) -> torch.Tensor:
        """
        Loss function for MultiVAE.

        Combines Binary Cross Entropy (BCE) and Kullback–Leibler Divergence (KLD)
        to form the Variational Autoencoder loss.

        Parameters:
            recon_x (torch.Tensor): Reconstructed input.
            x (torch.Tensor): Original input.
            mu (torch.Tensor): Mean from the latent space.
            logvar (torch.Tensor): Log variance from the latent space.
            anneal (float): Annealing factor for KLD.

        Returns:
            torch.Tensor: Calculated loss.
        """
        BCE = -torch.mean(torch.sum(F.log_softmax(recon_x, 1) * x, -1))
        KLD = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
        return BCE + anneal * KLD

    def mpr(self, I: torch.Tensor, R: torch.Tensor) -> float:
        num_users, num_items = R.shape
        total_interactions = torch.sum(R)

        # Create a tensor for percentile ranks (shape: num_items)
        percentile_ranks = (
            torch.arange(num_items, dtype=dtype, device=device) / num_items
        )

        # Use I to index into R and rearrange it, then multiply by the percentile ranks
        # Reshape and expand percentile ranks to match the shape of R
        # (shape of expanded percentile ranks: 1 x num_items)
        weighted_ranks = R.gather(1, I) * percentile_ranks.expand(num_users, -1)

        # Sum over all items and users, and normalize
        mpr = torch.sum(weighted_ranks) / total_interactions

        return mpr.item()

    def recommend_items(
        self,
        user_data: torch.Tensor,
        interacted_indices: Optional[torch.Tensor] = None,
        top_k: int = 10,
    ) -> List[int]:
        """
        Generate item recommendations for a user using the MultiVAE model.

        Args:
        model (MultiVAE): The trained MultiVAE model.
        user_data (torch.Tensor): The user-item interaction vector for a single user.
        interacted_indices (Optional[torch.Tensor]): A binary vector indicating items the user has already interacted with. If None, no filtering is applied.
        top_k (int): Number of top recommendations to return.

        Returns:
        List[int]: List of item indices recommended for the user.
        """
        self.eval()  # Ensure the model is in evaluation mode

        # Encode user data to latent space
        mu, logvar = self.encode(user_data)
        z = self.reparameterize(mu, logvar)

        # Decode the latent representation
        reconstructed_user_data = self.decode(z)

        # Convert to probabilities
        proba = torch.softmax(reconstructed_user_data, dim=1)

        # If interacted_indices is provided, mask out already interacted items
        if interacted_indices is not None:
            interaction_mask = torch.zeros(proba.size(), dtype=torch.bool, device=self.device)
            interaction_mask[0, interacted_indices] = True  # Assumes user_data is a single user batch
            proba.masked_fill_(interaction_mask, 0)

        # Get top k items that have the highest probability
        recommended_items = torch.topk(proba, top_k, dim=1)[1].squeeze().tolist()

        return recommended_items

In [9]:
users, items = user_interacted_with.shape

In [10]:
batch_size = 32

# train_set, val_set = train_test_split(R, test_size=0.2, random_state=42)

# # Convert the train and validation sets into TensorDataset objects
# train_dataset = TensorDataset(train_set)
# val_dataset = TensorDataset(val_set)

# # Create DataLoaders for batching
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

train_set = user_interacted_with
train_dataset = TensorDataset(train_set)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

eval_set = user_interacted_with
eval_dataset = TensorDataset(eval_set)
eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)

In [11]:
# Define dimensions for the encoder and decoder
encoder_dims = [user_interacted_with.shape[1], 800, 400]
decoder_dims = encoder_dims[::-1]  # Symmetric decoder

# Initialize the model
model = MultiVAE(encoder_dims, decoder_dims, dropout_rate=0.5, device=device, dtype=dtype)

In [12]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 1000  # Number of epochs
mprs = []  # Keep track of the MPRs
val_losses = []  # Keep track of the validation loss
train_losses = []  # Keep track of the training loss
for epoch in range(num_epochs):
    model.train()

    train_loss = 0.0
    for data in train_loader:
        data = data[0]
        # Assuming data is a batch of user-item interactions
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data)
        loss = model.loss(recon_batch, data, mu, logvar)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs}: Train Loss: {train_loss:.4f}"
    )

Epoch 1/1000: Train Loss: 5624.1055
Epoch 2/1000: Train Loss: 5607.3218
Epoch 3/1000: Train Loss: 5582.1230
Epoch 4/1000: Train Loss: 5539.0137
Epoch 5/1000: Train Loss: 5550.3882
Epoch 6/1000: Train Loss: 5592.0303
Epoch 7/1000: Train Loss: 5541.9160
Epoch 8/1000: Train Loss: 5496.5386
Epoch 9/1000: Train Loss: 5460.3896
Epoch 10/1000: Train Loss: 5438.9292
Epoch 11/1000: Train Loss: 5383.5469
Epoch 12/1000: Train Loss: 5342.7998
Epoch 13/1000: Train Loss: 5317.3174
Epoch 14/1000: Train Loss: 5117.1313
Epoch 15/1000: Train Loss: 5065.2104
Epoch 16/1000: Train Loss: 4921.5352
Epoch 17/1000: Train Loss: 4819.5454
Epoch 18/1000: Train Loss: 4760.5747
Epoch 19/1000: Train Loss: 4637.3257
Epoch 20/1000: Train Loss: 4414.9365
Epoch 21/1000: Train Loss: 4349.8472
Epoch 22/1000: Train Loss: 4275.3413
Epoch 23/1000: Train Loss: 4177.3701
Epoch 24/1000: Train Loss: 4088.6387
Epoch 25/1000: Train Loss: 4015.2397
Epoch 26/1000: Train Loss: 3991.8376
Epoch 27/1000: Train Loss: 3953.4673
Epoch 28/1

In [13]:
go.Figure(
    data=[
        go.Scatter(
            x=list(range(len(train_losses))),
            y=train_losses,
            name="Train Loss",
            mode="lines",
        ),
        go.Scatter(
            x=list(range(len(val_losses))),
            y=val_losses,
            name="Validation Loss",
            mode="lines",
        ),
    ],
    layout=go.Layout(
        title="Training and Validation Loss",
        xaxis=dict(title="Epoch"),
        yaxis=dict(title="Loss"),
    ),
)

In [14]:
# retrieve users latent representations (mean and variance)
usersnames = matrix_mf.get_usernames()

model.eval()
with torch.no_grad():
    mu, logvar = model.encode(user_interacted_with)
    z = model.reparameterize(mu, logvar)
    z = z.cpu().numpy()
    mu = mu.cpu().numpy()
    std = np.exp(0.5 * logvar.cpu().numpy())
    logvar = logvar.cpu().numpy()

print(z)
print(mu)
print(logvar)

[[ 0.38791552 -0.3356129  -0.36279678 ...  0.21034864 -0.652727
   0.73466605]
 [ 0.34460232  0.5198928  -0.23736435 ... -0.18044353 -0.08430137
  -0.05968325]
 [ 0.37243858 -0.27983394  0.36257616 ... -0.15310022  0.52561784
  -0.02193665]
 ...
 [-0.39632425  0.03191594 -0.00384138 ...  0.18850668 -0.51583374
  -0.65912867]
 [ 0.21524253 -0.02846021 -0.25904062 ...  0.29879734  0.26102725
  -0.415601  ]
 [-0.30007997 -0.04083337  0.30987248 ...  0.0309695  -0.00379538
   0.25391623]]
[[ 0.38791552 -0.3356129  -0.36279678 ...  0.21034864 -0.652727
   0.73466605]
 [ 0.34460232  0.5198928  -0.23736435 ... -0.18044353 -0.08430137
  -0.05968325]
 [ 0.37243858 -0.27983394  0.36257616 ... -0.15310022  0.52561784
  -0.02193665]
 ...
 [-0.39632425  0.03191594 -0.00384138 ...  0.18850668 -0.51583374
  -0.65912867]
 [ 0.21524253 -0.02846021 -0.25904062 ...  0.29879734  0.26102725
  -0.415601  ]
 [-0.30007997 -0.04083337  0.30987248 ...  0.0309695  -0.00379538
   0.25391623]]
[[-0.13494131 -0.228

In [15]:
# Define the number of points for grid, the width of the gaussian curves and color map
if encoder_dims[-1] == 2:
        # Plot 2D latent space using and add annotations with usernames
    fig = px.scatter(x=z[:, 0], y=z[:, 1], hover_name=usersnames, color=usersnames)
    fig.update_traces(marker=dict(size=10))
    fig.update_layout(title="2D Latent Space")
    # Show the usernames next to the points
    for i, username in enumerate(usersnames):
        fig.add_annotation(
            x=z[i, 0], y=z[i, 1], text=username, showarrow=False, xshift=-20, yshift=15
        )
    fig.show()
    
    plotting.plot_multivariate_gaussian_image_with_labels(mu, std, usersnames)

In [23]:
user = "paul"
user_idx = matrix_mf.usernames_to_ids([user])[0]
user_data = matrix_mf.get_user(user_idx).to(device).unsqueeze(0)
user_interacted_with = matrix_mf.user_interacted_with(user_idx)

recommended_items_ids = model.recommend_items(user_data, user_interacted_with, top_k=10)
recommended_items_ids

recommended_items = matrix_mf.ids_to_itemnames(recommended_items_ids)
df_recommended_items = df_matrix_mf[df_matrix_mf["id"].isin(recommended_items)]
df_recommended_items[spoti.PRETTY_PRINT_FEATURES]

,username,artists_names,name,release_year,popularity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
1516,owen,Hamza,Free YSL,2023,62,0.907,0.577,0.0574,0.063400,0.000010,0.1580,0.459,125.981,-8.146,186253,2023,62
1519,owen,Doja Cat,Agora Hills,2023,91,0.750,0.674,0.0970,0.228000,0.000089,0.1220,0.392,123.026,-6.128,265360,2023,91
1522,owen,BabySolo33,LnlyBby,2022,30,0.708,0.300,0.0752,0.410000,0.320000,0.1030,0.260,95.018,-12.861,174147,2022,30
1523,owen,BabySolo33,BFF <3,2022,28,0.845,0.280,0.1410,0.295000,0.001040,0.0896,0.527,120.031,-12.457,193654,2022,28
1528,owen,betcover!!,海豚少年 - アルバムバージョン,2019,21,0.681,0.708,0.0912,0.117000,0.011000,0.1000,0.324,100.386,-7.375,293733,2019,21
1533,owen,YG Pablo,22h22,2023,60,0.737,0.506,0.1310,0.747000,0.000130,0.1020,0.320,129.910,-11.587,191066,2023,60
1547,owen,Troye Sivan,One Of Your Girls,2023,88,0.629,0.654,0.0584,0.087800,0.006550,0.2200,0.799,93.034,-7.852,181481,2023,88
1551,owen,Dosseh,Macabre,2023,56,0.815,0.648,0.3030,0.559000,0.000000,0.1080,0.707,140.043,-6.465,208841,2023,56
1555,owen,YG Pablo,22h22,2023,60,0.737,0.506,0.1310,0.747000,0.000130,0.1020,0.320,129.910,-11.587,191066,2023,60
1556,owen,betcover!!,海豚少年 - アルバムバージョン,2019,21,0.681,0.708,0.0912,0.117000,0.011000,0.1000,0.324,100.386,-7.375,293733,2019,21
